

<div style="background: linear-gradient(120deg, #1ca9c9 0%, #40c4d6 60%, #7fe0e0 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Actividad 6:  Datos Faltantes (Imputación Multivariada)</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Laboratorio de Procesamiento de Datos</h3>
    </div>
</div>



Esta actividad consiste en aplicar y comparar dos **métodos de imputación multivariada** en un conjunto de datos numérico. Además de describir cómo funcionan, evaluarás qué tan bien recuperan valores que se ocultaron deliberadamente.

Se utilizará el conjunto **Diabetes** incluido en `scikit-learn` (`load_diabetes`). Contiene **442 observaciones** y **10 carácterísticas numéricas**, sus valores están escalados y las variables presentan relaciones entre sí. No se requiere descargar un archivo. La variable objetivo no se incluye en la imputación.

La primera celda carga las variables completas (`X_completo`), crea una máscara aleatoria reproducible y oculta cerca del 15 % de sus celdas (`X_incompleto`). Conserva `X_completo` y `mascara`: servirán para comparar las imputaciones con los valores reales que se ocultaron. En este ejercicio suponemos que los datos faltantes son **MCAR** (se generan al azar).


In [2]:
from sklearn.datasets import load_diabetes
import numpy as np

# Dataset completo para conservar los valores reales como referencia.
diabetes = load_diabetes(as_frame=True)
X_completo = diabetes.data

# Ocultamos aleatoriamente el 15 % de las celdas; la semilla hace reproducible el ejercicio.
rng = np.random.default_rng(42)
mascara = rng.random(X_completo.shape) < 0.15
X_incompleto = X_completo.mask(mascara)

print(f"Observaciones: {X_completo.shape[0]}")
print(f"Variables predictoras: {X_completo.shape[1]}")
print(f"Valores ocultos: {mascara.sum()} ({mascara.mean():.1%})")
X_incompleto.head()


Observaciones: 442
Variables predictoras: 10
Valores ocultos: 674 (15.2%)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,NaN,-0.034821,-0.043401,-0.002592,NaN,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,NaN,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,NaN,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,NaN,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,NaN


1. **KNNImputer.** Investiga y explica cómo `sklearn.impute.KNNImputer` estima un valor usando filas vecinas y la distancia euclidiana ajustada a las variables observadas en común. Aplica el método al dataset `X_incompleto`, prueba al menos dos valores de `n_neighbors` (argumento de entrada de KNNImputer) y explica cómo elegiste el mejor.

Imputa cada valor faltante en una fila tomando la media (ponderada o uniforme) de los valores correspondientes en sus $k$ vecinos más cercanos (k-Nearest Neighbors), pero para su calculo schikit learn aplica una ecuacion donde toma el numero de variables tota, el conjunto de variables donde ambas observaciones tienen valores (ninguna es NaN)
Una vez calculadas las distancias y seleccionados los k vecinos mas cercanos , el valor ausente de la variable se imputa calculando la media de los mas cercanos.


In [4]:
#Código aquí
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd

# Valores reales que fueron ocultados
valores_reales = X_completo.values[mascara]

# Probamos distintos valores de n_neighbors (incluyendo al menos dos, p. ej. 3, 5, 10 y 15)
vecinos_a_probar = [2, 3, 5, 10, 15]
resultados_knn = []

for k in vecinos_a_probar:
    imputer_knn = KNNImputer(n_neighbors=k)
    X_knn_arr = imputer_knn.fit_transform(X_incompleto)
    
    # Extraer únicamente los valores que se habían ocultado
    valores_imputados_knn = X_knn_arr[mascara]
    
    # Evaluar la calidad de recuperación frente a los valores reales
    rmse = np.sqrt(mean_squared_error(valores_reales, valores_imputados_knn))
    mae = mean_absolute_error(valores_reales, valores_imputados_knn)
    
    resultados_knn.append({
        "n_neighbors (k)": k,
        "RMSE": round(rmse, 5),
        "MAE": round(mae, 5)
    })

df_eval_knn = pd.DataFrame(resultados_knn)
print("EVALUACIÓN DE KNNImputer SEGÚN n_neighbors")
print(df_eval_knn.to_string(index=False))

EVALUACIÓN DE KNNImputer SEGÚN n_neighbors
 n_neighbors (k)    RMSE     MAE
               2 0.04211 0.03199
               3 0.03954 0.02999
               5 0.03706 0.02856
              10 0.03501 0.02718
              15 0.03466 0.02717


2. ### **IterativeImputer.** Investiga y explica cómo `sklearn.impute.IterativeImputer` estima cada variable incompleta a partir de las demás y repite el proceso por rondas. Aplica el método a `X_incompleto`.

### Como funciona sklearn.impute.iterativeimputer?

*IterativeImputer* se basa en la estrategia MICE (Multivariate Imputation by Chained Equations o Imputación Multivariada por Ecuaciones Encadenadas). Trata la imputación como una secuencia iterativa de problemas de regresión

* Imputación inicial previa: >    Todas las celdas faltantes se rellenan temporalmente con una estadística univariada simple (por defecto, la media de cada columna).

* Modelado columna por columna (en cadena): >    Para cada variable $j$ que contenga valores faltantes:

    * Se designa a la variable $j$ como la variable dependiente u objetivo ($y$).

    * Las restantes variables $X_{-j}$ actúan como características predictoras.

    * Se entrena un modelo de regresión (por defecto BayesianRidge) utilizando únicamente las filas donde la variable $j$ fue originalmente observada.

    * Se predicen y reemplazan los valores faltantes de $j$ empleando las predicciones del modelo sobre las demás variables.

* Rondas o iteraciones: >    Una pasada completa que actualiza todas las columnas incompletas constituye una ronda o ciclo.

* Convergencia: >    El proceso se repite sucesivamente por varias rondas (determinado por el parámetro max_iter, típicamente 10). En cada nueva ronda, las predicciones mejoran porque las variables predictoras ahora contienen imputaciones más refinadas de la ronda anterior, concluyendo cuando los cambios entre rondas caen por debajo de un umbral de tolerancia (tol) o se alcanza el número máximo de iteraciones.

MAE > Mean Absolut Error - Error absoluto medio

RMSE > Root Mean Squared Error - Raíz del error cuadratico medio

In [6]:
#Código aquí
# IterativeImputer es una característica experimental en scikit-learn, requiere habilitarse primero
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Instanciar y ajustar el modelo
imputer_iter = IterativeImputer(max_iter=15, random_state=42)
X_iter_arr = imputer_iter.fit_transform(X_incompleto)

# Extraer los valores imputados en las posiciones ocultas
valores_imputados_iter = X_iter_arr[mascara]

# Métricas de error frente a los datos reales
rmse_iter = np.sqrt(mean_squared_error(valores_reales, valores_imputados_iter))
mae_iter = mean_absolute_error(valores_reales, valores_imputados_iter)

print("EVALUACIÓN DE IterativeImputer")
print(f"RMSE: {rmse_iter:.5f}")
print(f"MAE:  {mae_iter:.5f}")

EVALUACIÓN DE IterativeImputer
RMSE: 0.02953
MAE:  0.02213


c:\Users\santi\OneDrive - ITESO\ITESO\Semestre 3\Procesamiento de Datos\Laboratorio_de_Procesamiento_de_Datos_02026\.venv\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [9]:
# Comparativa final en tabla
df_comparacion = pd.DataFrame([
    {
        "Método": "KNNImputer (k=3)",
        "RMSE": df_eval_knn.loc[df_eval_knn["n_neighbors (k)"] == 3, "RMSE"].values[0],
        "MAE": df_eval_knn.loc[df_eval_knn["n_neighbors (k)"] == 3, "MAE"].values[0]
    },
    {
        "Método": "KNNImputer k = 15",
        "RMSE": df_eval_knn.loc[df_eval_knn["n_neighbors (k)"] == 15, "RMSE"].values[0],
        "MAE": df_eval_knn.loc[df_eval_knn["n_neighbors (k)"] == 15, "MAE"].values[0]
    },
    {
        "Método": "IterativeImputer",
        "RMSE": round(rmse_iter, 5),
        "MAE": round(mae_iter, 5)
    }
])

print("COMPARACIÓN FINAL DE RENDIMIENTO")
print(df_comparacion.to_string(index=False))

COMPARACIÓN FINAL DE RENDIMIENTO
           Método    RMSE     MAE
 KNNImputer (k=3) 0.03954 0.02999
KNNImputer k = 15 0.03466 0.02717
 IterativeImputer 0.02953 0.02213


## Conclusion de la comparación

* Superioridad de IterativeImputer: >    *IterativeImputer* obtiene un menor error de reconstrucción ($\text{RMSE} \approx 0.0296$ frente a $\approx 0.0346$ del mejor KNN). Esto ocurre porque el dataset de Diabetes tiene correlaciones lineales significativas entre sus variables clínicas (especialmente entre las mediciones de suero sanguíneo s1 a s6), las cuales son capturadas de forma más efectiva por modelos de regresión lineal bayesiana que por la simple proximidad euclidiana local de vecinos.

* Complejidad y supuestos: >    *KNNImputer* no asume una relación funcional (lineal) entre variables, pero su costo computacional crece cuadráticamente con el número de filas al requerir el cálculo de matrices de distancias por pares.
    * IterativeImputer aprovecha la estructura de covarianza multivariada global del conjunto de datos mediante rondas progresivas, resultando especialmente preciso en datos continuos estandarizados.